In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "RandomForest",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            'base_LocalClustCoeff',
            'base_ComponentSize',
            'base_ComponentEdgeNodeRatio',
            'base_TriangleCount',
            'base_UniquePartners',
            'base_AvgNeighbourDegree',
            'base_BetweennessCentrality',
            'base_EigenvectorCentrality',
            'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_ProtStoichSum',
            # 'stoich_ProtAvgStoich',
            # 'stoich_ProtMaxStoich',
            # 'stoich_MinComplexSize',
            # 'stoich_MaxComplexSize',
            # 'stoich_AvgComplexSize',
            'stoich_WeightedTriangles',
            'stoich_AvgNeighbourDegreeStoich',
            'stoich_RangeComplexSize',
            'stoich_MedComplexSize',
            'stoich_MedianRatio',
            'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            # 'protein_MinComplexNodes',
            # 'protein_MaxComplexNodes',
            # 'protein_AvgComplexNodes',
            # 'protein_NormUniqueSum',
            # 'protein_AvgUniqueRatio',
            # 'protein_MaxUniqueRatio',
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            'pair_LocalClustCoeff',
            'pair_TriangleCount',
            'pair_ComponentSize',
            'pair_EigenvectorCentrality',
            'pair_BetweennessCentrality',
            'pair_KatzCentrality',
            'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features : {len(hypergraph_features)}")
    print(f"   Active pairwise features   : {len(pairwise_features)}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    # Statistical summary text
    with open(output_dir / 'statistical_summary.txt', 'w') as f:
        f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
        f.write("Task: Gene Essentiality\n")
        f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
        f.write(f"Number of splits: {stats['n_runs']}\n\n")
        f.write(f"Hypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
        f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
        f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
        f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
        f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
        f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
        f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
        f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")
    print("   Saved: statistical_summary.txt")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 09:06:41

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : RandomForest
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features : 22
   Active pairwise features   : 8

3. Running paired comparisons across 15 splits...

   Split  1/15... train=2658 (23.7% ess)  test=622 (23.2% ess)  |  Hyper: 0.4576, Pair: 0.4977, Diff: -0

<ipython-input-2-fb7f8d548138>:516: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_TriangleCount                  0.0348       0.0322
  2      stoich_WeightedTriangles            0.0340       0.0375
  3      base_AvgNeighbourDegree             0.0321       0.0141
  4      base_UniquePartners                 0.0176       0.0188
  5      protein_MedComplexNodes             0.0169       0.0135
  6      stoich_RangeComplexSize             0.0118       0.0162
  7      protein_RangeComplexNodes           0.0110       0.0129
  8      protein_RangeUniqueRatio            0.0100       0.0135
  9      protein_MedianUniqueRatio           0.0098       0.0079
  10     stoich_AvgNeighbourDegreeStoich     0.0083       0.0116

  Top 10 Pairwise Features:
  Rank   Feature                 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "RandomForest",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            'base_TriangleCount',
            'base_UniquePartners',
            'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_ProtStoichSum',
            # 'stoich_ProtAvgStoich',
            # 'stoich_ProtMaxStoich',
            # 'stoich_MinComplexSize',
            # 'stoich_MaxComplexSize',
            # 'stoich_AvgComplexSize',
            'stoich_WeightedTriangles',
            'stoich_AvgNeighbourDegreeStoich',
            'stoich_RangeComplexSize',
            'stoich_MedComplexSize',
            'stoich_MedianRatio',
            'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            # 'protein_MinComplexNodes',
            # 'protein_MaxComplexNodes',
            # 'protein_AvgComplexNodes',
            # 'protein_NormUniqueSum',
            # 'protein_AvgUniqueRatio',
            # 'protein_MaxUniqueRatio',
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            'pair_LocalClustCoeff',
            'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 09:14:46

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : RandomForest
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (17):
     - base_Degree
     - base_LocalClustCoeff
     - base_TriangleCount
     - base_UniquePartners
     - base_AvgNeighbourDegree
     - stoich_WeightedTriangles
     - stoich_AvgNeighbour

<ipython-input-3-6c3ab696f4b6>:516: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      stoich_WeightedTriangles            0.0482       0.0531
  2      base_AvgNeighbourDegree             0.0474       0.0112
  3      base_TriangleCount                  0.0456       0.0348
  4      base_UniquePartners                 0.0329       0.0305
  5      protein_MedianUniqueRatio           0.0206       0.0109
  6      protein_MedComplexNodes             0.0185       0.0154
  7      protein_RangeComplexNodes           0.0123       0.0118
  8      stoich_AvgNeighbourDegreeStoich     0.0119       0.0066
  9      stoich_RangeComplexSize             0.0116       0.0140
  10     protein_RangeUniqueRatio            0.0094       0.0091

  Top 10 Pairwise Features:
  Rank   Feature                 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "RandomForest",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            'base_TriangleCount',
            'base_UniquePartners',
            'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_ProtStoichSum',
            # 'stoich_ProtAvgStoich',
            # 'stoich_ProtMaxStoich',
            # 'stoich_MinComplexSize',
            # 'stoich_MaxComplexSize',
            # 'stoich_AvgComplexSize',
            # 'stoich_WeightedTriangles',
            # 'stoich_AvgNeighbourDegreeStoich',
            # 'stoich_RangeComplexSize',
            # 'stoich_MedComplexSize',
            # 'stoich_MedianRatio',
            # 'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            # 'protein_MinComplexNodes',
            # 'protein_MaxComplexNodes',
            # 'protein_AvgComplexNodes',
            # 'protein_NormUniqueSum',
            # 'protein_AvgUniqueRatio',
            # 'protein_MaxUniqueRatio',
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            'pair_LocalClustCoeff',
            'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 09:23:07

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : RandomForest
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (11):
     - base_Degree
     - base_LocalClustCoeff
     - base_TriangleCount
     - base_UniquePartners
     - base_AvgNeighbourDegree
     - protein_NumSubgroups
     - protein_AvgSubgroupSize

<ipython-input-4-4683ee01f24b>:516: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_AvgNeighbourDegree             0.0573       0.0184
  2      base_TriangleCount                  0.0446       0.0558
  3      base_UniquePartners                 0.0274       0.0372
  4      protein_RangeUniqueRatio            0.0224       0.0320
  5      protein_MedComplexNodes             0.0217       0.0217
  6      protein_RangeComplexNodes           0.0202       0.0221
  7      protein_AvgSubgroupSize             0.0197       0.0485
  8      protein_MedianUniqueRatio           0.0177       0.0165
  9      base_Degree                         0.0072       0.0129
  10     base_LocalClustCoeff                0.0059       0.0189

  Top 10 Pairwise Features:
  Rank   Feature                 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "RandomForest",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            'base_TriangleCount',
            'base_UniquePartners',
            'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_ProtStoichSum',
            # 'stoich_ProtAvgStoich',
            # 'stoich_ProtMaxStoich',
            # 'stoich_MinComplexSize',
            # 'stoich_MaxComplexSize',
            # 'stoich_AvgComplexSize',
            'stoich_WeightedTriangles',
            'stoich_AvgNeighbourDegreeStoich',
            'stoich_RangeComplexSize',
            'stoich_MedComplexSize',
            'stoich_MedianRatio',
            'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            # 'protein_MinComplexNodes',
            # 'protein_MaxComplexNodes',
            # 'protein_AvgComplexNodes',
            # 'protein_NormUniqueSum',
            # 'protein_AvgUniqueRatio',
            # 'protein_MaxUniqueRatio',
            # 'protein_NumSubgroups',
            # 'protein_AvgSubgroupSize',
            # 'protein_MedianUniqueRatio',
            # 'protein_RangeUniqueRatio',
            # 'protein_MedComplexNodes',
            # 'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            'pair_LocalClustCoeff',
            'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 09:28:46

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : RandomForest
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (11):
     - base_Degree
     - base_LocalClustCoeff
     - base_TriangleCount
     - base_UniquePartners
     - base_AvgNeighbourDegree
     - stoich_WeightedTriangles
     - stoich_AvgNeighbour

<ipython-input-5-90e83a18a178>:516: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_TriangleCount                  0.1133       0.0401
  2      base_UniquePartners                 0.0749       0.0311
  3      stoich_WeightedTriangles            0.0560       0.0605
  4      base_AvgNeighbourDegree             0.0473       0.0215
  5      stoich_RangeComplexSize             0.0162       0.0124
  6      stoich_AvgNeighbourDegreeStoich     0.0134       0.0112
  7      stoich_MedianRatio                  0.0125       0.0120
  8      base_Degree                         0.0071       0.0116
  9      base_LocalClustCoeff                0.0018       0.0063
  10     stoich_RangeRatio                   -0.0006       0.0040

  Top 10 Pairwise Features:
  Rank   Feature                

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./lightgbm/"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "LightGBM",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            'base_TriangleCount',
            'base_UniquePartners',
            'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_ProtStoichSum',
            # 'stoich_ProtAvgStoich',
            # 'stoich_ProtMaxStoich',
            # 'stoich_MinComplexSize',
            # 'stoich_MaxComplexSize',
            # 'stoich_AvgComplexSize',
            'stoich_WeightedTriangles',
            'stoich_AvgNeighbourDegreeStoich',
            'stoich_RangeComplexSize',
            'stoich_MedComplexSize',
            'stoich_MedianRatio',
            'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            # 'protein_MinComplexNodes',
            # 'protein_MaxComplexNodes',
            # 'protein_AvgComplexNodes',
            # 'protein_NormUniqueSum',
            # 'protein_AvgUniqueRatio',
            # 'protein_MaxUniqueRatio',
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            'pair_LocalClustCoeff',
            'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 09:35:40

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : LightGBM
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : lightgbm/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (17):
     - base_Degree
     - base_LocalClustCoeff
     - base_TriangleCount
     - base_UniquePartners
     - base_AvgNeighbourDegree
     - stoich_WeightedTriangles
     - stoich_AvgNeig

<ipython-input-6-828bf03f4bc8>:516: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_AvgNeighbourDegree             0.0783       0.0374
  2      base_TriangleCount                  0.0766       0.0577
  3      stoich_WeightedTriangles            0.0718       0.0834
  4      protein_MedianUniqueRatio           0.0412       0.0344
  5      base_UniquePartners                 0.0231       0.0369
  6      protein_MedComplexNodes             0.0170       0.0242
  7      stoich_RangeComplexSize             0.0110       0.0208
  8      stoich_MedianRatio                  0.0106       0.0122
  9      protein_AvgSubgroupSize             0.0105       0.0172
  10     protein_RangeComplexNodes           0.0090       0.0101

  Top 10 Pairwise Features:
  Rank   Feature                 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./randomforest/"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "RandomForest",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            'base_TriangleCount',
            'base_UniquePartners',
            # 'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_ProtStoichSum',
            # 'stoich_ProtAvgStoich',
            # 'stoich_ProtMaxStoich',
            # 'stoich_MinComplexSize',
            # 'stoich_MaxComplexSize',
            # 'stoich_AvgComplexSize',
            'stoich_WeightedTriangles',
            # 'stoich_AvgNeighbourDegreeStoich',
            'stoich_RangeComplexSize',
            'stoich_MedComplexSize',
            'stoich_MedianRatio',
            'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            # 'protein_MinComplexNodes',
            # 'protein_MaxComplexNodes',
            # 'protein_AvgComplexNodes',
            # 'protein_NormUniqueSum',
            # 'protein_AvgUniqueRatio',
            # 'protein_MaxUniqueRatio',
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            'pair_LocalClustCoeff',
            'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            # 'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 09:42:01

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : RandomForest
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : randomforest/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (15):
     - base_Degree
     - base_LocalClustCoeff
     - base_TriangleCount
     - base_UniquePartners
     - stoich_WeightedTriangles
     - stoich_RangeComplexSize
     - stoich

<ipython-input-7-0da015686478>:516: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      stoich_WeightedTriangles            0.0395       0.0386
  2      base_TriangleCount                  0.0356       0.0318
  3      base_UniquePartners                 0.0244       0.0263
  4      protein_MedianUniqueRatio           0.0149       0.0210
  5      stoich_MedianRatio                  0.0132       0.0158
  6      protein_MedComplexNodes             0.0126       0.0154
  7      stoich_RangeComplexSize             0.0125       0.0157
  8      protein_AvgSubgroupSize             0.0086       0.0135
  9      protein_RangeUniqueRatio            0.0086       0.0113
  10     base_LocalClustCoeff                0.0079       0.0096

  Top 10 Pairwise Features:
  Rank   Feature                 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./randomforest/base_features"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "RandomForest",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            'base_TriangleCount',
            'base_UniquePartners',
            # 'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_ProtStoichSum',
            # 'stoich_ProtAvgStoich',
            # 'stoich_ProtMaxStoich',
            # 'stoich_MinComplexSize',
            # 'stoich_MaxComplexSize',
            # 'stoich_AvgComplexSize',
            # 'stoich_WeightedTriangles',
            # 'stoich_AvgNeighbourDegreeStoich',
            # 'stoich_RangeComplexSize',
            # 'stoich_MedComplexSize',
            # 'stoich_MedianRatio',
            # 'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            # 'protein_MinComplexNodes',
            # 'protein_MaxComplexNodes',
            # 'protein_AvgComplexNodes',
            # 'protein_NormUniqueSum',
            # 'protein_AvgUniqueRatio',
            # 'protein_MaxUniqueRatio',
            # 'protein_NumSubgroups',
            # 'protein_AvgSubgroupSize',
            # 'protein_MedianUniqueRatio',
            # 'protein_RangeUniqueRatio',
            # 'protein_MedComplexNodes',
            # 'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            'pair_LocalClustCoeff',
            'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            # 'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 09:49:16

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : RandomForest
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : randomforest/base_features/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (4):
     - base_Degree
     - base_LocalClustCoeff
     - base_TriangleCount
     - base_UniquePartners
   Active pairwise features (3):
     - pair_Degree
     - pair

<ipython-input-8-280fb3f33eb6>:516: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_TriangleCount                  0.1314       0.0701
  2      base_UniquePartners                 0.1066       0.0561
  3      base_Degree                         0.0525       0.0461
  4      base_LocalClustCoeff                0.0182       0.0176

  Top 10 Pairwise Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      pair_Degree                         0.1194       0.0908
  2      pair_TriangleCount                  0.1144       0.0432
  3      pair_LocalClustCoeff                0.0577       0.0232

  Note: Higher = more important; negative = possible noise.
   Saved: feature_importance_c

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./randomforest/base_features"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "RandomForest",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            # 'base_TriangleCount',
            'base_UniquePartners',
            # 'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_ProtStoichSum',
            # 'stoich_ProtAvgStoich',
            # 'stoich_ProtMaxStoich',
            # 'stoich_MinComplexSize',
            # 'stoich_MaxComplexSize',
            # 'stoich_AvgComplexSize',
            # 'stoich_WeightedTriangles',
            # 'stoich_AvgNeighbourDegreeStoich',
            # 'stoich_RangeComplexSize',
            # 'stoich_MedComplexSize',
            # 'stoich_MedianRatio',
            # 'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            # 'protein_MinComplexNodes',
            # 'protein_MaxComplexNodes',
            # 'protein_AvgComplexNodes',
            # 'protein_NormUniqueSum',
            # 'protein_AvgUniqueRatio',
            # 'protein_MaxUniqueRatio',
            # 'protein_NumSubgroups',
            # 'protein_AvgSubgroupSize',
            # 'protein_MedianUniqueRatio',
            # 'protein_RangeUniqueRatio',
            # 'protein_MedComplexNodes',
            # 'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            # 'pair_LocalClustCoeff',
            # 'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            # 'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 09:56:45

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : RandomForest
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : randomforest/base_features/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (2):
     - base_Degree
     - base_UniquePartners
   Active pairwise features (1):
     - pair_Degree

3. Running paired comparisons across 15 splits...

   Split  1/1

<ipython-input-9-4d9904961ce3>:516: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_UniquePartners                 0.2488       0.1104
  2      base_Degree                         0.0953       0.0576

  Top 10 Pairwise Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      pair_Degree                         0.2145       0.0987

  Note: Higher = more important; negative = possible noise.
   Saved: feature_importance_comparison.png

7. Saving outputs...
   Saved: split_results.csv
   Saved: hypergraph_predictions.csv
   Saved: pairwise_predictions.csv
   Saved: hypergraph_feature_importance.csv
   Saved: pairwise_feature_importance.csv

  COMPLETE

Total time: 274.0s (4.6 m

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./randomforest/base_features"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "RandomForest",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            # 'base_TriangleCount',
            # 'base_UniquePartners',
            # 'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_ProtStoichSum',
            # 'stoich_ProtAvgStoich',
            # 'stoich_ProtMaxStoich',
            # 'stoich_MinComplexSize',
            # 'stoich_MaxComplexSize',
            # 'stoich_AvgComplexSize',
            # 'stoich_WeightedTriangles',
            # 'stoich_AvgNeighbourDegreeStoich',
            # 'stoich_RangeComplexSize',
            # 'stoich_MedComplexSize',
            # 'stoich_MedianRatio',
            # 'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            # 'protein_MinComplexNodes',
            # 'protein_MaxComplexNodes',
            # 'protein_AvgComplexNodes',
            # 'protein_NormUniqueSum',
            # 'protein_AvgUniqueRatio',
            # 'protein_MaxUniqueRatio',
            # 'protein_NumSubgroups',
            # 'protein_AvgSubgroupSize',
            # 'protein_MedianUniqueRatio',
            # 'protein_RangeUniqueRatio',
            # 'protein_MedComplexNodes',
            # 'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            # 'pair_LocalClustCoeff',
            # 'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            # 'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 10:13:54

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : RandomForest
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : randomforest/base_features/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (1):
     - base_Degree
   Active pairwise features (1):
     - pair_Degree

3. Running paired comparisons across 15 splits...

   Split  1/15... train=2658 (23.7% ess)

<ipython-input-10-9feee63a627b>:516: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_Degree                         0.0415       0.0287

  Top 10 Pairwise Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      pair_Degree                         0.2145       0.0987

  Note: Higher = more important; negative = possible noise.
   Saved: feature_importance_comparison.png

7. Saving outputs...
   Saved: split_results.csv
   Saved: hypergraph_predictions.csv
   Saved: pairwise_predictions.csv
   Saved: hypergraph_feature_importance.csv
   Saved: pairwise_feature_importance.csv

  COMPLETE

Total time: 264.5s (4.4 min)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./randomforest/base_features"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "RandomForest",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            # 'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            # 'base_TriangleCount',
            'base_UniquePartners',
            # 'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            'stoich_ProtStoichSum',
            'stoich_ProtAvgStoich',
            'stoich_ProtMaxStoich',
            'stoich_MinComplexSize',
            'stoich_MaxComplexSize',
            'stoich_AvgComplexSize',
            # 'stoich_WeightedTriangles',
            # 'stoich_AvgNeighbourDegreeStoich',
            # 'stoich_RangeComplexSize',
            # 'stoich_MedComplexSize',
            # 'stoich_MedianRatio',
            # 'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            # 'protein_MinComplexNodes',
            # 'protein_MaxComplexNodes',
            # 'protein_AvgComplexNodes',
            # 'protein_NormUniqueSum',
            # 'protein_AvgUniqueRatio',
            # 'protein_MaxUniqueRatio',
            # 'protein_NumSubgroups',
            # 'protein_AvgSubgroupSize',
            # 'protein_MedianUniqueRatio',
            # 'protein_RangeUniqueRatio',
            # 'protein_MedComplexNodes',
            # 'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            # 'pair_LocalClustCoeff',
            # 'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            # 'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 10:23:32

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : RandomForest
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : randomforest/base_features/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (7):
     - base_UniquePartners
     - stoich_ProtStoichSum
     - stoich_ProtAvgStoich
     - stoich_ProtMaxStoich
     - stoich_MinComplexSize
     - stoich_MaxComple

<ipython-input-12-85d296e49799>:516: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_UniquePartners                 0.1642       0.0831
  2      stoich_MaxComplexSize               0.0635       0.0574
  3      stoich_ProtAvgStoich                0.0325       0.0218
  4      stoich_AvgComplexSize               0.0195       0.0357
  5      stoich_ProtStoichSum                0.0167       0.0200
  6      stoich_ProtMaxStoich                0.0125       0.0209
  7      stoich_MinComplexSize               0.0023       0.0169

  Top 10 Pairwise Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      pair_Degree                         0.2145       0.0987

  Note: Higher = more imp

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./randomforest/base_features"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "RandomForest",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            # 'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            # 'base_TriangleCount',
            'base_UniquePartners',
            # 'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_ProtStoichSum',
            # 'stoich_ProtAvgStoich',
            # 'stoich_ProtMaxStoich',
            # 'stoich_MinComplexSize',
            # 'stoich_MaxComplexSize',
            # 'stoich_AvgComplexSize',
            # 'stoich_WeightedTriangles',
            # 'stoich_AvgNeighbourDegreeStoich',
            'stoich_RangeComplexSize',
            'stoich_MedComplexSize',
            'stoich_MedianRatio',
            'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            # 'protein_MinComplexNodes',
            # 'protein_MaxComplexNodes',
            # 'protein_AvgComplexNodes',
            # 'protein_NormUniqueSum',
            # 'protein_AvgUniqueRatio',
            # 'protein_MaxUniqueRatio',
            # 'protein_NumSubgroups',
            # 'protein_AvgSubgroupSize',
            # 'protein_MedianUniqueRatio',
            # 'protein_RangeUniqueRatio',
            # 'protein_MedComplexNodes',
            # 'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            # 'pair_LocalClustCoeff',
            # 'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            # 'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 10:28:41

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : RandomForest
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : randomforest/base_features/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (5):
     - base_UniquePartners
     - stoich_RangeComplexSize
     - stoich_MedComplexSize
     - stoich_MedianRatio
     - stoich_RangeRatio
   Active pairwise featur

<ipython-input-13-af1c6ba6fc23>:516: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_UniquePartners                 0.1936       0.0829
  2      stoich_MedianRatio                  0.0682       0.0184
  3      stoich_RangeComplexSize             0.0516       0.0354
  4      stoich_MedComplexSize               0.0282       0.0415
  5      stoich_RangeRatio                   0.0057       0.0093

  Top 10 Pairwise Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      pair_Degree                         0.2145       0.0987

  Note: Higher = more important; negative = possible noise.
   Saved: feature_importance_comparison.png

7. Saving outputs...
   Saved: split_results.csv
 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./randomforest/base_features"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "RandomForest",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            'base_TriangleCount',
            'base_UniquePartners',
            # 'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_ProtStoichSum',
            # 'stoich_ProtAvgStoich',
            # 'stoich_ProtMaxStoich',
            # 'stoich_MinComplexSize',
            # 'stoich_MaxComplexSize',
            # 'stoich_AvgComplexSize',
            'stoich_WeightedTriangles',
            # 'stoich_AvgNeighbourDegreeStoich',
            'stoich_RangeComplexSize',
            'stoich_MedComplexSize',
            'stoich_MedianRatio',
            'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            # 'protein_MinComplexNodes',
            # 'protein_MaxComplexNodes',
            # 'protein_AvgComplexNodes',
            # 'protein_NormUniqueSum',
            # 'protein_AvgUniqueRatio',
            # 'protein_MaxUniqueRatio',
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            # 'pair_LocalClustCoeff',
            # 'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            # 'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 10:42:05

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : RandomForest
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : randomforest/base_features/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (14):
     - base_Degree
     - base_TriangleCount
     - base_UniquePartners
     - stoich_WeightedTriangles
     - stoich_RangeComplexSize
     - stoich_MedComplexSiz

<ipython-input-14-7e9d91d4598b>:516: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      stoich_WeightedTriangles            0.0426       0.0508
  2      base_TriangleCount                  0.0355       0.0383
  3      base_UniquePartners                 0.0225       0.0279
  4      protein_MedianUniqueRatio           0.0222       0.0108
  5      protein_MedComplexNodes             0.0198       0.0145
  6      stoich_RangeComplexSize             0.0170       0.0212
  7      stoich_MedianRatio                  0.0155       0.0114
  8      protein_NumSubgroups                0.0148       0.0181
  9      protein_RangeUniqueRatio            0.0142       0.0169
  10     stoich_MedComplexSize               0.0129       0.0284

  Top 10 Pairwise Features:
  Rank   Feature                 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./randomforest/base_features"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "RandomForest",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            'base_TriangleCount',
            'base_UniquePartners',
            # 'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_ProtStoichSum',
            # 'stoich_ProtAvgStoich',
            # 'stoich_ProtMaxStoich',
            # 'stoich_MinComplexSize',
            # 'stoich_MaxComplexSize',
            # 'stoich_AvgComplexSize',
            # 'stoich_WeightedTriangles',
            # 'stoich_AvgNeighbourDegreeStoich',
            # 'stoich_RangeComplexSize',
            # 'stoich_MedComplexSize',
            # 'stoich_MedianRatio',
            # 'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            # 'protein_MinComplexNodes',
            # 'protein_MaxComplexNodes',
            # 'protein_AvgComplexNodes',
            # 'protein_NormUniqueSum',
            # 'protein_AvgUniqueRatio',
            # 'protein_MaxUniqueRatio',
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            # 'pair_LocalClustCoeff',
            # 'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            # 'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 11:35:33

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : RandomForest
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : randomforest/base_features/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (9):
     - base_Degree
     - base_TriangleCount
     - base_UniquePartners
     - protein_NumSubgroups
     - protein_AvgSubgroupSize
     - protein_MedianUniqueRatio

<ipython-input-15-90135ebe6d7b>:516: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_TriangleCount                  0.0400       0.0397
  2      base_UniquePartners                 0.0390       0.0331
  3      protein_MedComplexNodes             0.0260       0.0181
  4      protein_MedianUniqueRatio           0.0162       0.0229
  5      protein_RangeComplexNodes           0.0157       0.0173
  6      protein_RangeUniqueRatio            0.0153       0.0140
  7      protein_AvgSubgroupSize             0.0136       0.0287
  8      protein_NumSubgroups                0.0094       0.0174
  9      base_Degree                         0.0058       0.0158

  Top 10 Pairwise Features:
  Rank   Feature                             Mean         Std       
  ---------------------------

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./randomforest/base_features"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "RandomForest",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            # 'base_TriangleCount',
            'base_UniquePartners',
            # 'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_ProtStoichSum',
            # 'stoich_ProtAvgStoich',
            # 'stoich_ProtMaxStoich',
            # 'stoich_MinComplexSize',
            # 'stoich_MaxComplexSize',
            # 'stoich_AvgComplexSize',
            # 'stoich_WeightedTriangles',
            # 'stoich_AvgNeighbourDegreeStoich',
            # 'stoich_RangeComplexSize',
            # 'stoich_MedComplexSize',
            # 'stoich_MedianRatio',
            # 'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            # 'protein_MinComplexNodes',
            # 'protein_MaxComplexNodes',
            # 'protein_AvgComplexNodes',
            # 'protein_NormUniqueSum',
            # 'protein_AvgUniqueRatio',
            # 'protein_MaxUniqueRatio',
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            # 'pair_LocalClustCoeff',
            # 'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            # 'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 11:42:04

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : RandomForest
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : randomforest/base_features/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (8):
     - base_Degree
     - base_UniquePartners
     - protein_NumSubgroups
     - protein_AvgSubgroupSize
     - protein_MedianUniqueRatio
     - protein_RangeUniqu

<ipython-input-16-4daacd86bf52>:516: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_UniquePartners                 0.0696       0.0524
  2      protein_MedianUniqueRatio           0.0235       0.0180
  3      protein_AvgSubgroupSize             0.0179       0.0410
  4      protein_MedComplexNodes             0.0177       0.0148
  5      protein_RangeUniqueRatio            0.0161       0.0155
  6      protein_RangeComplexNodes           0.0145       0.0177
  7      protein_NumSubgroups                0.0105       0.0272
  8      base_Degree                         0.0022       0.0116

  Top 10 Pairwise Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      pair_Degree      

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./randomforest/base_features"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "RandomForest",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            'base_ComponentSize',
            'base_ComponentEdgeNodeRatio',
            'base_TriangleCount',
            'base_UniquePartners',
            'base_AvgNeighbourDegree',
            'base_BetweennessCentrality',
            'base_EigenvectorCentrality',
            'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            'stoich_ProtStoichSum',
            'stoich_ProtAvgStoich',
            'stoich_ProtMaxStoich',
            'stoich_MinComplexSize',
            'stoich_MaxComplexSize',
            'stoich_AvgComplexSize',
            'stoich_WeightedTriangles',
            'stoich_AvgNeighbourDegreeStoich',
            # 'stoich_RangeComplexSize',
            # 'stoich_MedComplexSize',
            # 'stoich_MedianRatio',
            # 'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            'protein_MinComplexNodes',
            'protein_MaxComplexNodes',
            'protein_AvgComplexNodes',
            'protein_NormUniqueSum',
            'protein_AvgUniqueRatio',
            'protein_MaxUniqueRatio',
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            # 'protein_MedianUniqueRatio',
            # 'protein_RangeUniqueRatio',
            # 'protein_MedComplexNodes',
            # 'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            'pair_LocalClustCoeff',
            'pair_TriangleCount',
            'pair_ComponentSize',
            'pair_EigenvectorCentrality',
            'pair_BetweennessCentrality',
            'pair_KatzCentrality',
            'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 12:15:35

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : RandomForest
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : randomforest/base_features/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (25):
     - base_Degree
     - base_ComponentSize
     - base_ComponentEdgeNodeRatio
     - base_TriangleCount
     - base_UniquePartners
     - base_AvgNeighbourDegre

<ipython-input-17-e81a9e87878b>:516: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_TriangleCount                  0.0312       0.0325
  2      stoich_WeightedTriangles            0.0311       0.0344
  3      base_AvgNeighbourDegree             0.0249       0.0098
  4      protein_MaxComplexNodes             0.0204       0.0149
  5      base_UniquePartners                 0.0199       0.0205
  6      protein_AvgComplexNodes             0.0129       0.0144
  7      base_KatzCentrality                 0.0118       0.0152
  8      stoich_AvgNeighbourDegreeStoich     0.0068       0.0099
  9      protein_AvgUniqueRatio              0.0066       0.0083
  10     stoich_ProtStoichSum                0.0048       0.0092

  Top 10 Pairwise Features:
  Rank   Feature                 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./randomforest/base_features"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "RandomForest",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            'base_TriangleCount',
            'base_UniquePartners',
            'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            'stoich_WeightedTriangles',
            'stoich_AvgNeighbourDegreeStoich',
            'stoich_RangeComplexSize',
            'stoich_MedComplexSize',
            'stoich_MedianRatio',
            'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            'pair_LocalClustCoeff',
            'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 12:24:34

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : RandomForest
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : randomforest/base_features/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (16):
     - base_Degree
     - base_TriangleCount
     - base_UniquePartners
     - base_AvgNeighbourDegree
     - stoich_WeightedTriangles
     - stoich_AvgNeighbourD

<ipython-input-18-c2ed85f07d1c>:504: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_AvgNeighbourDegree             0.0526       0.0088
  2      base_TriangleCount                  0.0477       0.0424
  3      stoich_WeightedTriangles            0.0378       0.0442
  4      protein_MedianUniqueRatio           0.0245       0.0110
  5      base_UniquePartners                 0.0233       0.0229
  6      protein_MedComplexNodes             0.0184       0.0132
  7      stoich_RangeComplexSize             0.0171       0.0162
  8      stoich_AvgNeighbourDegreeStoich     0.0171       0.0124
  9      protein_RangeComplexNodes           0.0170       0.0189
  10     protein_AvgSubgroupSize             0.0105       0.0235

  Top 10 Pairwise Features:
  Rank   Feature                 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./lightgbm/two_hop_features"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "LightGBM",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            'base_TriangleCount',
            'base_UniquePartners',
            'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            'stoich_WeightedTriangles',
            'stoich_AvgNeighbourDegreeStoich',
            'stoich_RangeComplexSize',
            'stoich_MedComplexSize',
            'stoich_MedianRatio',
            'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            'pair_LocalClustCoeff',
            'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 12:32:28

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : LightGBM
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : lightgbm/two_hop_features/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (16):
     - base_Degree
     - base_TriangleCount
     - base_UniquePartners
     - base_AvgNeighbourDegree
     - stoich_WeightedTriangles
     - stoich_AvgNeighbourDegree

<ipython-input-20-894a9e77ef81>:504: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_AvgNeighbourDegree             0.0860       0.0366
  2      base_TriangleCount                  0.0781       0.0622
  3      stoich_WeightedTriangles            0.0700       0.0792
  4      protein_MedianUniqueRatio           0.0458       0.0434
  5      base_UniquePartners                 0.0199       0.0395
  6      protein_MedComplexNodes             0.0171       0.0260
  7      protein_RangeUniqueRatio            0.0145       0.0170
  8      protein_AvgSubgroupSize             0.0120       0.0169
  9      stoich_MedianRatio                  0.0104       0.0114
  10     protein_NumSubgroups                0.0088       0.0140

  Top 10 Pairwise Features:
  Rank   Feature                 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./xgboost/two_hop_features"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "XGBoost",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            'base_TriangleCount',
            'base_UniquePartners',
            'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            'stoich_WeightedTriangles',
            'stoich_AvgNeighbourDegreeStoich',
            'stoich_RangeComplexSize',
            'stoich_MedComplexSize',
            'stoich_MedianRatio',
            'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            'pair_LocalClustCoeff',
            'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 12:37:21

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : XGBoost
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : xgboost/two_hop_features/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (16):
     - base_Degree
     - base_TriangleCount
     - base_UniquePartners
     - base_AvgNeighbourDegree
     - stoich_WeightedTriangles
     - stoich_AvgNeighbourDegreeSt

<ipython-input-21-e6e5fbcfeef1>:504: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_AvgNeighbourDegree             0.1070       0.0294
  2      base_TriangleCount                  0.0890       0.0500
  3      stoich_WeightedTriangles            0.0516       0.0464
  4      protein_MedianUniqueRatio           0.0372       0.0496
  5      protein_MedComplexNodes             0.0331       0.0506
  6      base_UniquePartners                 0.0201       0.0264
  7      base_Degree                         0.0185       0.0375
  8      protein_RangeUniqueRatio            0.0182       0.0225
  9      stoich_RangeComplexSize             0.0095       0.0193
  10     stoich_AvgNeighbourDegreeStoich     0.0091       0.0191

  Top 10 Pairwise Features:
  Rank   Feature                 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./xgboost/one_hop_features"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "XGBoost",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            # 'base_TriangleCount',
            'base_UniquePartners',
            # 'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_WeightedTriangles',
            # 'stoich_AvgNeighbourDegreeStoich',
            'stoich_RangeComplexSize',
            'stoich_MedComplexSize',
            'stoich_MedianRatio',
            'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            # 'pair_LocalClustCoeff',
            # 'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            # 'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 12:58:39

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : XGBoost
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : xgboost/one_hop_features/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (12):
     - base_Degree
     - base_UniquePartners
     - stoich_RangeComplexSize
     - stoich_MedComplexSize
     - stoich_MedianRatio
     - stoich_RangeRatio
     - prote

<ipython-input-22-e11e27511179>:504: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_UniquePartners                 0.0912       0.0650
  2      protein_MedComplexNodes             0.0632       0.0537
  3      protein_MedianUniqueRatio           0.0485       0.0409
  4      protein_RangeUniqueRatio            0.0397       0.0216
  5      stoich_RangeComplexSize             0.0270       0.0227
  6      stoich_MedianRatio                  0.0213       0.0114
  7      base_Degree                         0.0177       0.0287
  8      stoich_MedComplexSize               0.0120       0.0188
  9      protein_NumSubgroups                0.0074       0.0114
  10     protein_AvgSubgroupSize             0.0060       0.0081

  Top 10 Pairwise Features:
  Rank   Feature                 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./lightgbm/one_hop_features"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "LightGBM",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            # 'base_TriangleCount',
            'base_UniquePartners',
            # 'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_WeightedTriangles',
            # 'stoich_AvgNeighbourDegreeStoich',
            'stoich_RangeComplexSize',
            'stoich_MedComplexSize',
            'stoich_MedianRatio',
            'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            # 'pair_LocalClustCoeff',
            # 'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            # 'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 13:09:07

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : LightGBM
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : lightgbm/one_hop_features/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (12):
     - base_Degree
     - base_UniquePartners
     - stoich_RangeComplexSize
     - stoich_MedComplexSize
     - stoich_MedianRatio
     - stoich_RangeRatio
     - pro

<ipython-input-23-d6862d5062e7>:504: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_UniquePartners                 0.1162       0.0706
  2      protein_MedianUniqueRatio           0.0433       0.0677
  3      protein_MedComplexNodes             0.0424       0.0607
  4      stoich_RangeComplexSize             0.0244       0.0208
  5      protein_RangeUniqueRatio            0.0225       0.0226
  6      stoich_MedianRatio                  0.0165       0.0157
  7      stoich_MedComplexSize               0.0149       0.0313
  8      protein_AvgSubgroupSize             0.0088       0.0282
  9      protein_RangeComplexNodes           0.0083       0.0074
  10     base_Degree                         0.0060       0.0142

  Top 10 Pairwise Features:
  Rank   Feature                 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./randomforest/one_hop_features"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "RandomForest",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            # 'base_TriangleCount',
            'base_UniquePartners',
            # 'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_WeightedTriangles',
            # 'stoich_AvgNeighbourDegreeStoich',
            'stoich_RangeComplexSize',
            'stoich_MedComplexSize',
            'stoich_MedianRatio',
            'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            # 'pair_LocalClustCoeff',
            # 'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            # 'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 15:02:26

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : RandomForest
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : randomforest/one_hop_features/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (12):
     - base_Degree
     - base_UniquePartners
     - stoich_RangeComplexSize
     - stoich_MedComplexSize
     - stoich_MedianRatio
     - stoich_RangeRatio
  

<ipython-input-24-b6240b8e8c25>:504: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_UniquePartners                 0.0599       0.0546
  2      protein_MedianUniqueRatio           0.0349       0.0229
  3      protein_MedComplexNodes             0.0286       0.0217
  4      stoich_RangeComplexSize             0.0251       0.0205
  5      stoich_MedianRatio                  0.0156       0.0070
  6      protein_RangeUniqueRatio            0.0121       0.0071
  7      protein_RangeComplexNodes           0.0109       0.0162
  8      stoich_MedComplexSize               0.0083       0.0188
  9      protein_NumSubgroups                0.0077       0.0072
  10     protein_AvgSubgroupSize             0.0073       0.0176

  Top 10 Pairwise Features:
  Rank   Feature                 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./randomforest/all_features"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "RandomForest",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            'base_LocalClustCoeff',
            'base_ComponentSize',
            'base_ComponentEdgeNodeRatio',
            'base_TriangleCount',
            'base_UniquePartners',
            'base_AvgNeighbourDegree',
            'base_BetweennessCentrality',
            'base_EigenvectorCentrality',
            'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            'stoich_WeightedTriangles',
            'stoich_AvgNeighbourDegreeStoich',
            'stoich_RangeComplexSize',
            'stoich_MedComplexSize',
            'stoich_MedianRatio',
            'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            'pair_LocalClustCoeff',
            'pair_TriangleCount',
            'pair_ComponentSize',
            'pair_EigenvectorCentrality',
            'pair_BetweennessCentrality',
            'pair_KatzCentrality',
            'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 16:01:26

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : RandomForest
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : randomforest/all_features/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (22):
     - base_Degree
     - base_LocalClustCoeff
     - base_ComponentSize
     - base_ComponentEdgeNodeRatio
     - base_TriangleCount
     - base_UniquePartners
  

<ipython-input-25-cedc308b1e73>:504: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_TriangleCount                  0.0348       0.0322
  2      stoich_WeightedTriangles            0.0340       0.0375
  3      base_AvgNeighbourDegree             0.0321       0.0141
  4      base_UniquePartners                 0.0176       0.0188
  5      protein_MedComplexNodes             0.0169       0.0135
  6      stoich_RangeComplexSize             0.0118       0.0162
  7      protein_RangeComplexNodes           0.0110       0.0129
  8      protein_RangeUniqueRatio            0.0100       0.0135
  9      protein_MedianUniqueRatio           0.0098       0.0079
  10     stoich_AvgNeighbourDegreeStoich     0.0083       0.0116

  Top 10 Pairwise Features:
  Rank   Feature                 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./lightgbm/all_features"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "LightGBM",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            'base_LocalClustCoeff',
            'base_ComponentSize',
            'base_ComponentEdgeNodeRatio',
            'base_TriangleCount',
            'base_UniquePartners',
            'base_AvgNeighbourDegree',
            'base_BetweennessCentrality',
            'base_EigenvectorCentrality',
            'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            'stoich_WeightedTriangles',
            'stoich_AvgNeighbourDegreeStoich',
            'stoich_RangeComplexSize',
            'stoich_MedComplexSize',
            'stoich_MedianRatio',
            'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            'pair_LocalClustCoeff',
            'pair_TriangleCount',
            'pair_ComponentSize',
            'pair_EigenvectorCentrality',
            'pair_BetweennessCentrality',
            'pair_KatzCentrality',
            'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 16:10:04

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : LightGBM
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : lightgbm/all_features/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (22):
     - base_Degree
     - base_LocalClustCoeff
     - base_ComponentSize
     - base_ComponentEdgeNodeRatio
     - base_TriangleCount
     - base_UniquePartners
     - bas

<ipython-input-26-cf4d2af2b46a>:504: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_AvgNeighbourDegree             0.0781       0.0290
  2      base_TriangleCount                  0.0512       0.0407
  3      stoich_WeightedTriangles            0.0410       0.0364
  4      protein_MedComplexNodes             0.0303       0.0427
  5      protein_MedianUniqueRatio           0.0263       0.0269
  6      base_ComponentSize                  0.0249       0.0192
  7      base_ComponentEdgeNodeRatio         0.0217       0.0506
  8      base_BetweennessCentrality          0.0144       0.0106
  9      base_UniquePartners                 0.0139       0.0195
  10     base_KatzCentrality                 0.0110       0.0109

  Top 10 Pairwise Features:
  Rank   Feature                 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./xgboost/all_features"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "XGBoost",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            'base_LocalClustCoeff',
            'base_ComponentSize',
            'base_ComponentEdgeNodeRatio',
            'base_TriangleCount',
            'base_UniquePartners',
            'base_AvgNeighbourDegree',
            'base_BetweennessCentrality',
            'base_EigenvectorCentrality',
            'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            'stoich_WeightedTriangles',
            'stoich_AvgNeighbourDegreeStoich',
            'stoich_RangeComplexSize',
            'stoich_MedComplexSize',
            'stoich_MedianRatio',
            'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            'pair_LocalClustCoeff',
            'pair_TriangleCount',
            'pair_ComponentSize',
            'pair_EigenvectorCentrality',
            'pair_BetweennessCentrality',
            'pair_KatzCentrality',
            'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 16:17:42

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : XGBoost
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : xgboost/all_features/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (22):
     - base_Degree
     - base_LocalClustCoeff
     - base_ComponentSize
     - base_ComponentEdgeNodeRatio
     - base_TriangleCount
     - base_UniquePartners
     - base_

<ipython-input-27-c9c245842e96>:504: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_TriangleCount                  0.0842       0.0500
  2      base_AvgNeighbourDegree             0.0629       0.0263
  3      base_EigenvectorCentrality          0.0464       0.0624
  4      protein_MedianUniqueRatio           0.0311       0.0451
  5      stoich_WeightedTriangles            0.0264       0.0260
  6      protein_MedComplexNodes             0.0225       0.0275
  7      base_ComponentSize                  0.0201       0.0184
  8      base_UniquePartners                 0.0164       0.0367
  9      base_KatzCentrality                 0.0141       0.0175
  10     base_BetweennessCentrality          0.0132       0.0085

  Top 10 Pairwise Features:
  Rank   Feature                 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./xgboost/two_hop_no_stoich"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "XGBoost",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            'base_TriangleCount',
            'base_UniquePartners',
            'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_WeightedTriangles',
            # 'stoich_AvgNeighbourDegreeStoich',
            # 'stoich_RangeComplexSize',
            # 'stoich_MedComplexSize',
            # 'stoich_MedianRatio',
            # 'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            'pair_LocalClustCoeff',
            'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 16:40:31

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : XGBoost
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : xgboost/two_hop_no_stoich/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (10):
     - base_Degree
     - base_TriangleCount
     - base_UniquePartners
     - base_AvgNeighbourDegree
     - protein_NumSubgroups
     - protein_AvgSubgroupSize
     -

<ipython-input-29-e95efe95c7bd>:504: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_AvgNeighbourDegree             0.1356       0.0273
  2      base_TriangleCount                  0.0829       0.0353
  3      base_UniquePartners                 0.0443       0.0389
  4      protein_MedianUniqueRatio           0.0434       0.0569
  5      protein_RangeUniqueRatio            0.0376       0.0402
  6      protein_MedComplexNodes             0.0257       0.0300
  7      base_Degree                         0.0207       0.0345
  8      protein_RangeComplexNodes           0.0098       0.0091
  9      protein_AvgSubgroupSize             0.0063       0.0150
  10     protein_NumSubgroups                -0.0026       0.0065

  Top 10 Pairwise Features:
  Rank   Feature                

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./lightgbm/two_hop_no_stoich"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "LightGBM",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            'base_TriangleCount',
            'base_UniquePartners',
            'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_WeightedTriangles',
            # 'stoich_AvgNeighbourDegreeStoich',
            # 'stoich_RangeComplexSize',
            # 'stoich_MedComplexSize',
            # 'stoich_MedianRatio',
            # 'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            'pair_LocalClustCoeff',
            'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 16:48:48

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : LightGBM
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : lightgbm/two_hop_no_stoich/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (10):
     - base_Degree
     - base_TriangleCount
     - base_UniquePartners
     - base_AvgNeighbourDegree
     - protein_NumSubgroups
     - protein_AvgSubgroupSize
    

<ipython-input-30-857f2eea87e2>:504: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_AvgNeighbourDegree             0.1124       0.0397
  2      base_TriangleCount                  0.0918       0.0508
  3      protein_MedianUniqueRatio           0.0324       0.0271
  4      protein_RangeUniqueRatio            0.0315       0.0438
  5      base_UniquePartners                 0.0312       0.0542
  6      protein_MedComplexNodes             0.0139       0.0247
  7      base_Degree                         0.0113       0.0137
  8      protein_RangeComplexNodes           0.0105       0.0144
  9      protein_AvgSubgroupSize             0.0078       0.0222
  10     protein_NumSubgroups                0.0028       0.0106

  Top 10 Pairwise Features:
  Rank   Feature                 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./randomforest/two_hop_no_stoich"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "RandomForest",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            'base_TriangleCount',
            'base_UniquePartners',
            'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_WeightedTriangles',
            # 'stoich_AvgNeighbourDegreeStoich',
            # 'stoich_RangeComplexSize',
            # 'stoich_MedComplexSize',
            # 'stoich_MedianRatio',
            # 'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            'pair_LocalClustCoeff',
            'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 16:53:22

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : RandomForest
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : randomforest/two_hop_no_stoich/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (10):
     - base_Degree
     - base_TriangleCount
     - base_UniquePartners
     - base_AvgNeighbourDegree
     - protein_NumSubgroups
     - protein_AvgSubgroupS

<ipython-input-31-5036bbf3c592>:504: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_AvgNeighbourDegree             0.0677       0.0188
  2      base_UniquePartners                 0.0426       0.0431
  3      base_TriangleCount                  0.0324       0.0352
  4      protein_MedianUniqueRatio           0.0251       0.0173
  5      protein_RangeComplexNodes           0.0219       0.0182
  6      protein_RangeUniqueRatio            0.0197       0.0160
  7      protein_MedComplexNodes             0.0183       0.0114
  8      protein_AvgSubgroupSize             0.0180       0.0380
  9      base_Degree                         0.0057       0.0136
  10     protein_NumSubgroups                0.0054       0.0164

  Top 10 Pairwise Features:
  Rank   Feature                 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./randomforest/one_hop_no_stoich"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "RandomForest",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            # 'base_TriangleCount',
            'base_UniquePartners',
            # 'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_WeightedTriangles',
            # 'stoich_AvgNeighbourDegreeStoich',
            # 'stoich_RangeComplexSize',
            # 'stoich_MedComplexSize',
            # 'stoich_MedianRatio',
            # 'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            # 'pair_LocalClustCoeff',
            # 'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            # 'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 16:59:53

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : RandomForest
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : randomforest/one_hop_no_stoich/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (8):
     - base_Degree
     - base_UniquePartners
     - protein_NumSubgroups
     - protein_AvgSubgroupSize
     - protein_MedianUniqueRatio
     - protein_RangeU

<ipython-input-32-8ba7d53bd526>:504: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_UniquePartners                 0.0696       0.0524
  2      protein_MedianUniqueRatio           0.0235       0.0180
  3      protein_AvgSubgroupSize             0.0179       0.0410
  4      protein_MedComplexNodes             0.0177       0.0148
  5      protein_RangeUniqueRatio            0.0161       0.0155
  6      protein_RangeComplexNodes           0.0145       0.0177
  7      protein_NumSubgroups                0.0105       0.0272
  8      base_Degree                         0.0022       0.0116

  Top 10 Pairwise Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      pair_Degree      

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./lightgbm/one_hop_no_stoich"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "LightGBM",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            # 'base_TriangleCount',
            'base_UniquePartners',
            # 'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_WeightedTriangles',
            # 'stoich_AvgNeighbourDegreeStoich',
            # 'stoich_RangeComplexSize',
            # 'stoich_MedComplexSize',
            # 'stoich_MedianRatio',
            # 'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            # 'pair_LocalClustCoeff',
            # 'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            # 'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 17:06:59

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : LightGBM
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : lightgbm/one_hop_no_stoich/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (8):
     - base_Degree
     - base_UniquePartners
     - protein_NumSubgroups
     - protein_AvgSubgroupSize
     - protein_MedianUniqueRatio
     - protein_RangeUniqueRat

<ipython-input-33-4a92f776ea01>:504: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_UniquePartners                 0.1082       0.0433
  2      protein_MedianUniqueRatio           0.0419       0.0820
  3      protein_RangeUniqueRatio            0.0361       0.0320
  4      base_Degree                         0.0135       0.0235
  5      protein_MedComplexNodes             0.0125       0.0409
  6      protein_RangeComplexNodes           0.0107       0.0102
  7      protein_AvgSubgroupSize             0.0088       0.0152
  8      protein_NumSubgroups                0.0004       0.0072

  Top 10 Pairwise Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      pair_Degree      

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./xgboost/one_hop_no_stoich"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "XGBoost",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            # 'base_TriangleCount',
            'base_UniquePartners',
            # 'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_WeightedTriangles',
            # 'stoich_AvgNeighbourDegreeStoich',
            # 'stoich_RangeComplexSize',
            # 'stoich_MedComplexSize',
            # 'stoich_MedianRatio',
            # 'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            'protein_MedianUniqueRatio',
            'protein_RangeUniqueRatio',
            'protein_MedComplexNodes',
            'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            # 'pair_LocalClustCoeff',
            # 'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            # 'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 17:12:01

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : XGBoost
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : xgboost/one_hop_no_stoich/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (8):
     - base_Degree
     - base_UniquePartners
     - protein_NumSubgroups
     - protein_AvgSubgroupSize
     - protein_MedianUniqueRatio
     - protein_RangeUniqueRatio

<ipython-input-34-20ba2c6eafb3>:504: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_UniquePartners                 0.0955       0.0472
  2      protein_RangeUniqueRatio            0.0467       0.0273
  3      protein_MedianUniqueRatio           0.0423       0.0436
  4      protein_MedComplexNodes             0.0418       0.0524
  5      base_Degree                         0.0159       0.0371
  6      protein_RangeComplexNodes           0.0051       0.0061
  7      protein_NumSubgroups                0.0037       0.0099
  8      protein_AvgSubgroupSize             0.0013       0.0207

  Top 10 Pairwise Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      pair_Degree      

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./xgboost/one_hop_no_unique"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "XGBoost",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            # 'base_TriangleCount',
            'base_UniquePartners',
            # 'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_WeightedTriangles',
            # 'stoich_AvgNeighbourDegreeStoich',
            'stoich_RangeComplexSize',
            'stoich_MedComplexSize',
            'stoich_MedianRatio',
            'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            # 'protein_MedianUniqueRatio',
            # 'protein_RangeUniqueRatio',
            # 'protein_MedComplexNodes',
            # 'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            # 'pair_LocalClustCoeff',
            # 'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            # 'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 17:22:56

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : XGBoost
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : xgboost/one_hop_no_unique/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (8):
     - base_Degree
     - base_UniquePartners
     - stoich_RangeComplexSize
     - stoich_MedComplexSize
     - stoich_MedianRatio
     - stoich_RangeRatio
     - prote

<ipython-input-35-13e2711899f8>:504: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_UniquePartners                 0.2493       0.1002
  2      base_Degree                         0.0420       0.0280
  3      stoich_RangeComplexSize             0.0270       0.0191
  4      stoich_MedianRatio                  0.0148       0.0181
  5      stoich_MedComplexSize               0.0119       0.0174
  6      protein_AvgSubgroupSize             0.0075       0.0176
  7      protein_NumSubgroups                0.0053       0.0104
  8      stoich_RangeRatio                   0.0011       0.0058

  Top 10 Pairwise Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      pair_Degree      

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./lightgbm/one_hop_no_unique"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "LightGBM",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            # 'base_TriangleCount',
            'base_UniquePartners',
            # 'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_WeightedTriangles',
            # 'stoich_AvgNeighbourDegreeStoich',
            'stoich_RangeComplexSize',
            'stoich_MedComplexSize',
            'stoich_MedianRatio',
            'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            # 'protein_MedianUniqueRatio',
            # 'protein_RangeUniqueRatio',
            # 'protein_MedComplexNodes',
            # 'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            # 'pair_LocalClustCoeff',
            # 'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            # 'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 17:37:50

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : LightGBM
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : lightgbm/one_hop_no_unique/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (8):
     - base_Degree
     - base_UniquePartners
     - stoich_RangeComplexSize
     - stoich_MedComplexSize
     - stoich_MedianRatio
     - stoich_RangeRatio
     - pro

<ipython-input-36-153e761529c3>:504: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_UniquePartners                 0.2421       0.0927
  2      base_Degree                         0.0460       0.0285
  3      stoich_MedComplexSize               0.0452       0.0527
  4      stoich_RangeComplexSize             0.0425       0.0495
  5      stoich_MedianRatio                  0.0188       0.0146
  6      protein_NumSubgroups                0.0087       0.0180
  7      protein_AvgSubgroupSize             0.0084       0.0151
  8      stoich_RangeRatio                   -0.0012       0.0079

  Top 10 Pairwise Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      pair_Degree     

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./randomforest/one_hop_no_unique"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "RandomForest",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            # 'base_TriangleCount',
            'base_UniquePartners',
            # 'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            # 'stoich_WeightedTriangles',
            # 'stoich_AvgNeighbourDegreeStoich',
            'stoich_RangeComplexSize',
            'stoich_MedComplexSize',
            'stoich_MedianRatio',
            'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            # 'protein_MedianUniqueRatio',
            # 'protein_RangeUniqueRatio',
            # 'protein_MedComplexNodes',
            # 'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            # 'pair_LocalClustCoeff',
            # 'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            # 'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 17:43:00

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : RandomForest
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : randomforest/one_hop_no_unique/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (8):
     - base_Degree
     - base_UniquePartners
     - stoich_RangeComplexSize
     - stoich_MedComplexSize
     - stoich_MedianRatio
     - stoich_RangeRatio
  

<ipython-input-37-e47f0ecbc756>:504: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_UniquePartners                 0.1489       0.0623
  2      stoich_RangeComplexSize             0.0399       0.0249
  3      stoich_MedianRatio                  0.0396       0.0197
  4      protein_AvgSubgroupSize             0.0339       0.0326
  5      base_Degree                         0.0089       0.0128
  6      protein_NumSubgroups                0.0064       0.0060
  7      stoich_RangeRatio                   0.0059       0.0085
  8      stoich_MedComplexSize               0.0050       0.0170

  Top 10 Pairwise Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      pair_Degree      

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./randomforest/two_hop_no_unique"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "RandomForest",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            'base_TriangleCount',
            'base_UniquePartners',
            'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            'stoich_WeightedTriangles',
            'stoich_AvgNeighbourDegreeStoich',
            'stoich_RangeComplexSize',
            'stoich_MedComplexSize',
            'stoich_MedianRatio',
            'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            # 'protein_MedianUniqueRatio',
            # 'protein_RangeUniqueRatio',
            # 'protein_MedComplexNodes',
            # 'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            'pair_LocalClustCoeff',
            'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 17:48:55

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : RandomForest
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : randomforest/two_hop_no_unique/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (12):
     - base_Degree
     - base_TriangleCount
     - base_UniquePartners
     - base_AvgNeighbourDegree
     - stoich_WeightedTriangles
     - stoich_AvgNeighb

<ipython-input-38-5990c8e6d7c7>:504: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_TriangleCount                  0.1017       0.0338
  2      base_UniquePartners                 0.0596       0.0299
  3      stoich_WeightedTriangles            0.0481       0.0472
  4      base_AvgNeighbourDegree             0.0478       0.0190
  5      stoich_RangeComplexSize             0.0168       0.0153
  6      stoich_AvgNeighbourDegreeStoich     0.0167       0.0103
  7      stoich_MedianRatio                  0.0123       0.0103
  8      protein_NumSubgroups                0.0062       0.0067
  9      protein_AvgSubgroupSize             0.0048       0.0103
  10     base_Degree                         0.0026       0.0069

  Top 10 Pairwise Features:
  Rank   Feature                 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./lightgbm/two_hop_no_unique"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "LightGBM",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            'base_TriangleCount',
            'base_UniquePartners',
            'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            'stoich_WeightedTriangles',
            'stoich_AvgNeighbourDegreeStoich',
            'stoich_RangeComplexSize',
            'stoich_MedComplexSize',
            'stoich_MedianRatio',
            'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            # 'protein_MedianUniqueRatio',
            # 'protein_RangeUniqueRatio',
            # 'protein_MedComplexNodes',
            # 'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            'pair_LocalClustCoeff',
            'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 17:55:12

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : LightGBM
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : lightgbm/two_hop_no_unique/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (12):
     - base_Degree
     - base_TriangleCount
     - base_UniquePartners
     - base_AvgNeighbourDegree
     - stoich_WeightedTriangles
     - stoich_AvgNeighbourDegre

<ipython-input-39-5dbb2169b574>:504: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_TriangleCount                  0.1928       0.0574
  2      base_AvgNeighbourDegree             0.0956       0.0324
  3      stoich_WeightedTriangles            0.0861       0.0771
  4      base_UniquePartners                 0.0287       0.0243
  5      stoich_RangeComplexSize             0.0185       0.0259
  6      stoich_AvgNeighbourDegreeStoich     0.0103       0.0132
  7      base_Degree                         0.0080       0.0116
  8      stoich_MedianRatio                  0.0077       0.0095
  9      protein_NumSubgroups                0.0073       0.0247
  10     protein_AvgSubgroupSize             0.0070       0.0141

  Top 10 Pairwise Features:
  Rank   Feature                 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, average_precision_score
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# =======================================================
# Plotting Style Configuration
# =======================================================
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 20
})

# =======================================================
# CONFIGURATION
# =======================================================
CONFIG = {
    # --- Paths ---
    "DATA_DIR": Path("../../data/lookup_tables"),
    "BASE_OUTPUT_DIR": Path("./xgboost/two_hop_no_unique"),

    # --- File Names ---
    "SPLITS_FILE":           "protein_splits_all_strat.csv",
    "PROTEIN_FEATURES_FILE": "protein_noimpute_hypergraph_features.csv",
    "PAIRWISE_FEATURES_FILE":"pairwise_features_noself.csv",

    # --- Model ---
    # Options: "RandomForest" | "LightGBM" | "XGBoost"
    "MODEL_TYPE": "XGBoost",

    # --- Fixed settings ---
    "RANDOM_STATE": 42,
    "N_SPLITS_CV":  5,

    # --- Model-Specific Hyperparameter Grids for GridSearchCV ---
    "PARAM_GRIDS": {
        "RandomForest": {
            'n_estimators':      [80, 100, 200],
            'max_depth':         [None, 5, 10],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced']
        },
        "LightGBM": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'num_leaves':    [30, 50, 100],
            'class_weight':  ['balanced']
        },
        "XGBoost": {
            'n_estimators':  [80, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [None, 5, 10],
            'subsample':     [0.75, 0.8, 1.0],
            # scale_pos_weight is set automatically from training data (see tune_and_train_model)
        }
    },

    # --- Feature Selection ---
    # Comment/uncomment individual features to include or exclude them.
    "FEATURES": {
        "HYPERGRAPH": [
            # --- Base / native hypergraph metrics ---
            'base_Degree',
            # 'base_LocalClustCoeff',
            # 'base_ComponentSize',
            # 'base_ComponentEdgeNodeRatio',
            'base_TriangleCount',
            'base_UniquePartners',
            'base_AvgNeighbourDegree',
            # 'base_BetweennessCentrality',
            # 'base_EigenvectorCentrality',
            # 'base_KatzCentrality',

            # --- Stoichiometry-based metrics ---
            'stoich_WeightedTriangles',
            'stoich_AvgNeighbourDegreeStoich',
            'stoich_RangeComplexSize',
            'stoich_MedComplexSize',
            'stoich_MedianRatio',
            'stoich_RangeRatio',

            # --- Protein-participation metrics ---
            'protein_NumSubgroups',
            'protein_AvgSubgroupSize',
            # 'protein_MedianUniqueRatio',
            # 'protein_RangeUniqueRatio',
            # 'protein_MedComplexNodes',
            # 'protein_RangeComplexNodes',
        ],
        "PAIRWISE": [
            'pair_Degree',
            'pair_LocalClustCoeff',
            'pair_TriangleCount',
            # 'pair_ComponentSize',
            # 'pair_EigenvectorCentrality',
            # 'pair_BetweennessCentrality',
            # 'pair_KatzCentrality',
            'pair_AvgNeighborDegree',
        ]
    }
}


# =======================================================
# DATA LOADING
# =======================================================

def load_all_features() -> pd.DataFrame:
    """Loads hypergraph and pairwise feature CSVs and merges them on ProteinId."""
    print("1. Loading feature data...")

    hg_df   = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PROTEIN_FEATURES_FILE"])
    pair_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["PAIRWISE_FEATURES_FILE"])

    combined = pd.merge(hg_df, pair_df, on='ProteinId', how='inner')

    print(f"   Hypergraph features shape : {hg_df.shape}")
    print(f"   Pairwise features shape   : {pair_df.shape}")
    print(f"   Combined shape            : {combined.shape}")
    return combined


def load_splits() -> pd.DataFrame:
    """
    Loads the pre-assigned family-level splits file.

    Expected columns:
        split_index   — integer 1..N identifying which split
        UniProt_AC    — protein identifier (matches ProteinId in feature files)
        split         — 'train' or 'test'
        protein_label — 'Essential' | 'Non-essential' | 'Unknown'
        label_mask    — bool; False for Unknown proteins (excluded from metrics)
    """
    print("2. Loading pre-assigned splits...")
    splits_df = pd.read_csv(CONFIG["DATA_DIR"] / CONFIG["SPLITS_FILE"])

    # Rename to match feature file key
    splits_df = splits_df.rename(columns={'UniProt_AC': 'ProteinId'})

    # Encode binary target: Essential=1, Non-essential=0; Unknown kept as NaN
    label_map = {'Essential': 1, 'Non-essential': 0}
    splits_df['target'] = splits_df['protein_label'].map(label_map)

    n_splits = splits_df['split_index'].nunique()
    print(f"   Splits file rows  : {len(splits_df)}")
    print(f"   Unique proteins   : {splits_df['ProteinId'].nunique()}")
    print(f"   Number of splits  : {n_splits}")

    labelled = splits_df[splits_df['label_mask']].drop_duplicates('ProteinId')
    n_ess = (labelled['target'] == 1).sum()
    n_tot = len(labelled)
    print(f"   Labelled proteins : {n_tot}  ({100*n_ess/n_tot:.1f}% essential)")

    return splits_df


# =======================================================
# MODEL TRAINING & EVALUATION
# =======================================================

def tune_and_train_model(X_train: pd.DataFrame, y_train: pd.Series):
    """Hyperparameter search + fit.  Returns (best_estimator, best_params)."""
    model_type = CONFIG["MODEL_TYPE"]

    if model_type == "RandomForest":
        base_model = RandomForestClassifier(random_state=CONFIG["RANDOM_STATE"])
        param_grid = CONFIG["PARAM_GRIDS"]["RandomForest"]

    elif model_type == "LightGBM":
        base_model = LGBMClassifier(
            random_state=CONFIG["RANDOM_STATE"], n_jobs=1, verbose=-1
        )
        param_grid = CONFIG["PARAM_GRIDS"]["LightGBM"]

    elif model_type == "XGBoost":
        pos = int((y_train == 1).sum())
        neg = int((y_train == 0).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0
        base_model = XGBClassifier(
            random_state=CONFIG["RANDOM_STATE"],
            n_jobs=-1,
            verbosity=0,
            eval_metric='logloss',
            scale_pos_weight=spw
        )
        param_grid = CONFIG["PARAM_GRIDS"]["XGBoost"]

    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")

    gs = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=CONFIG["N_SPLITS_CV"],
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_


def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
    """Returns PR-AUC, F1 for the positive class, and predicted probabilities."""
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test, y_pred,
        target_names=['Non-Essential', 'Essential'],
        output_dict=True
    )

    return {
        'pr_auc':       average_precision_score(y_test, y_pred_proba),
        'f1':           report['Essential']['f1-score'],
        'y_pred_proba': y_pred_proba
    }


def compute_permutation_importance(
    model, X_test: pd.DataFrame, y_test: pd.Series, n_repeats: int = 10
) -> Dict[str, float]:
    """Permutation importance scored by average_precision (PR-AUC drop)."""
    result = permutation_importance(
        model, X_test, y_test,
        scoring='average_precision',
        n_repeats=n_repeats,
        random_state=CONFIG["RANDOM_STATE"],
        n_jobs=-1
    )
    return dict(zip(X_test.columns, result.importances_mean))


# =======================================================
# PER-SPLIT RUNNER
# =======================================================

def run_split(
    split_idx: int,
    merged_df: pd.DataFrame,
    splits_df: pd.DataFrame,
    hypergraph_features: List[str],
    pairwise_features: List[str]
) -> Dict:
    """
    Runs both hypergraph and pairwise models for a single pre-assigned split.

    merged_df   — feature matrix (ProteinId + all feature columns)
    splits_df   — full splits table (all split indices)
    Returns a results dict with PR-AUC, F1, importances, and per-protein predictions.
    """
    # --- Extract this split's assignments ---
    split_mask = splits_df['split_index'] == split_idx
    split_info = splits_df[split_mask][['ProteinId', 'split', 'target', 'label_mask']].copy()

    # Merge features with split assignments
    df = pd.merge(merged_df, split_info, on='ProteinId', how='inner')

    # Only use labelled proteins for training/evaluation
    labelled_df = df[df['label_mask']].copy()

    train_df = labelled_df[labelled_df['split'] == 'train']
    test_df  = labelled_df[labelled_df['split'] == 'test']

    y_train = train_df['target'].astype(int)
    y_test  = test_df['target'].astype(int)

    results = {
        'split_index':  split_idx,
        'n_train':      len(train_df),
        'n_test':       len(test_df),
        'train_ess_pct': 100 * y_train.mean(),
        'test_ess_pct':  100 * y_test.mean(),
    }

    # --- Hypergraph model ---
    X_hyper_train = train_df[hypergraph_features]
    X_hyper_test  = test_df[hypergraph_features]

    hyper_model, hyper_params = tune_and_train_model(X_hyper_train, y_train)
    hyper_eval = evaluate_model(hyper_model, X_hyper_test, y_test)

    results['hypergraph_pr_auc']    = hyper_eval['pr_auc']
    results['hypergraph_f1']        = hyper_eval['f1']
    results['hypergraph_best_params'] = hyper_params
    results['hypergraph_importance'] = compute_permutation_importance(
        hyper_model, X_hyper_test, y_test
    )

    # Store per-protein predictions (hypergraph)
    hyper_preds = test_df[['ProteinId']].copy()
    hyper_preds['split_index']       = split_idx
    hyper_preds['true_label']        = y_test.values
    hyper_preds['hyper_pred_proba']  = hyper_eval['y_pred_proba']
    results['hypergraph_predictions'] = hyper_preds

    # --- Pairwise model ---
    X_pair_train = train_df[pairwise_features]
    X_pair_test  = test_df[pairwise_features]

    pair_model, pair_params = tune_and_train_model(X_pair_train, y_train)
    pair_eval = evaluate_model(pair_model, X_pair_test, y_test)

    results['pairwise_pr_auc']    = pair_eval['pr_auc']
    results['pairwise_f1']        = pair_eval['f1']
    results['pairwise_best_params'] = pair_params
    results['pairwise_importance'] = compute_permutation_importance(
        pair_model, X_pair_test, y_test
    )

    # Store per-protein predictions (pairwise)
    pair_preds = test_df[['ProteinId']].copy()
    pair_preds['split_index']      = split_idx
    pair_preds['true_label']       = y_test.values
    pair_preds['pair_pred_proba']  = pair_eval['y_pred_proba']
    results['pairwise_predictions'] = pair_preds

    # Difference
    results['pr_auc_diff'] = results['hypergraph_pr_auc'] - results['pairwise_pr_auc']
    results['f1_diff']     = results['hypergraph_f1']     - results['pairwise_f1']

    return results


# =======================================================
# STATISTICAL COMPARISON
# =======================================================

def run_sign_test_comparison(all_results: List[Dict]) -> Dict:
    """Sign test (binomial) on paired PR-AUC wins/losses across splits."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    n_wins_hyper = int(np.sum(diffs > 0))
    n_wins_pair  = int(np.sum(diffs < 0))
    n_ties       = int(np.sum(diffs == 0))
    n_valid      = n_wins_hyper + n_wins_pair

    if n_valid > 0:
        p_greater   = binomtest(n_wins_hyper, n_valid, 0.5, alternative='greater').pvalue
        p_two_sided = binomtest(n_wins_hyper, n_valid, 0.5, alternative='two-sided').pvalue
    else:
        p_greater = p_two_sided = 1.0

    return {
        'n_runs':                   len(all_results),
        'hypergraph_pr_auc_mean':   float(np.mean(hyper_vals)),
        'hypergraph_pr_auc_std':    float(np.std(hyper_vals)),
        'pairwise_pr_auc_mean':     float(np.mean(pair_vals)),
        'pairwise_pr_auc_std':      float(np.std(pair_vals)),
        'mean_difference':          float(np.mean(diffs)),
        'std_difference':           float(np.std(diffs)),
        'hypergraph_wins':          n_wins_hyper,
        'pairwise_wins':            n_wins_pair,
        'ties':                     n_ties,
        'sign_test_p_greater':      p_greater,
        'sign_test_p_two_sided':    p_two_sided,
    }


# =======================================================
# FEATURE IMPORTANCE AGGREGATION
# =======================================================

def aggregate_feature_importance(
    all_results: List[Dict], representation: str
) -> pd.DataFrame:
    """
    Aggregates permutation importance across all splits.
    representation: 'hypergraph' or 'pairwise'
    """
    key = f'{representation}_importance'
    records = []
    for r in all_results:
        if key in r:
            for feat, imp in r[key].items():
                records.append({'split_index': r['split_index'],
                                'feature': feat, 'importance': imp})

    if not records:
        return pd.DataFrame()

    imp_df = pd.DataFrame(records)
    agg_df = (
        imp_df.groupby('feature')['importance']
        .agg(mean='mean', std='std', median='median',
             min='min', max='max', n_splits='count')
        .reset_index()
        .sort_values('mean', ascending=False)
        .reset_index(drop=True)
    )
    agg_df['rank'] = range(1, len(agg_df) + 1)
    return agg_df


# =======================================================
# PRINTING
# =======================================================

def print_statistical_summary(stats: Dict):
    print(f"\n{'='*70}")
    print("  STATISTICAL COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"{'='*70}")
    print(f"\n  Number of splits: {stats['n_runs']}")
    print(f"\n  {'Metric':<25} {'Hypergraph':<25} {'Pairwise':<20}")
    print(f"  {'-'*70}")
    print(f"  {'PR-AUC Mean ± Std':<25} "
          f"{stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}      "
          f"{stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}")
    print(f"\n  Mean Difference (Hyper - Pair): "
          f"{stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}")
    print(f"\n  Win/Loss Record:")
    n = stats['n_runs']
    print(f"    Hypergraph wins : {stats['hypergraph_wins']}/{n} "
          f"({100*stats['hypergraph_wins']/n:.1f}%)")
    print(f"    Pairwise wins   : {stats['pairwise_wins']}/{n} "
          f"({100*stats['pairwise_wins']/n:.1f}%)")
    print(f"    Ties            : {stats['ties']}/{n}")
    print(f"\n  Sign Test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}")
    print(f"  Sign Test p (two-sided)        : {stats['sign_test_p_two_sided']:.6f}")
    print(f"{'='*70}")


def print_feature_importance_summary(
    hyper_imp_df: pd.DataFrame, pair_imp_df: pd.DataFrame, top_n: int = 10
):
    print(f"\n{'='*70}")
    print("  FEATURE IMPORTANCE (Permutation — PR-AUC drop)")
    print(f"{'='*70}")
    for label, df in [("Hypergraph", hyper_imp_df), ("Pairwise", pair_imp_df)]:
        print(f"\n  Top {top_n} {label} Features:")
        print(f"  {'Rank':<6} {'Feature':<35} {'Mean':<12} {'Std':<10}")
        print(f"  {'-'*65}")
        for _, row in df.head(top_n).iterrows():
            print(f"  {int(row['rank']):<6} {row['feature']:<35} "
                  f"{row['mean']:.4f}       {row['std']:.4f}")
    print(f"\n  Note: Higher = more important; negative = possible noise.")
    print(f"{'='*70}")


# =======================================================
# PLOTTING
# =======================================================

def plot_paired_comparison(all_results: List[Dict], stats: Dict, output_dir: Path):
    """Three-panel comparison plot: histogram of diffs, scatter, and boxplot."""
    hyper_vals = np.array([r['hypergraph_pr_auc'] for r in all_results])
    pair_vals  = np.array([r['pairwise_pr_auc']   for r in all_results])
    diffs      = hyper_vals - pair_vals

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: histogram of differences
    ax1 = axes[0]
    ax1.hist(diffs, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(0, color='red',   linestyle='--', linewidth=2, label='No difference')
    ax1.axvline(diffs.mean(), color='green', linestyle='-', linewidth=2,
                label=f'Mean diff: {diffs.mean():.4f}')
    ax1.set_xlabel('PR-AUC Difference (Hypergraph − Pairwise)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Paired Differences')
    ax1.legend()

    # Panel 2: scatter (one point per split)
    ax2 = axes[1]
    ax2.scatter(pair_vals, hyper_vals, alpha=0.7, s=60, zorder=3)
    lo = min(pair_vals.min(), hyper_vals.min()) - 0.02
    hi = max(pair_vals.max(), hyper_vals.max()) + 0.02
    ax2.plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='y = x')
    ax2.set_xlabel('Pairwise PR-AUC')
    ax2.set_ylabel('Hypergraph PR-AUC')
    ax2.set_title('Paired Comparison — One Point per Split')
    ax2.set_xlim(lo, hi)
    ax2.set_ylim(lo, hi)
    ax2.set_aspect('equal')
    above = int(np.sum(hyper_vals > pair_vals))
    below = int(np.sum(hyper_vals < pair_vals))
    ax2.text(0.95, 0.05,
             f'Hypergraph wins: {above}\nPairwise wins: {below}',
             transform=ax2.transAxes, ha='right', va='bottom',
             bbox=dict(facecolor='lightgreen', alpha=0.5))

    # Panel 3: boxplot
    ax3 = axes[2]
    bp = ax3.boxplot([pair_vals, hyper_vals],
                     labels=['Pairwise', 'Hypergraph'],
                     patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgray')
    bp['boxes'][1].set_facecolor('steelblue')
    ax3.set_ylabel('PR-AUC')
    ax3.set_title('Distribution Comparison')
    rng = np.random.default_rng(0)
    for i, data in enumerate([pair_vals, hyper_vals]):
        x = rng.normal(i + 1, 0.04, size=len(data))
        ax3.scatter(x, data, alpha=0.4, s=20, color='black')

    plt.tight_layout()
    plt.savefig(output_dir / 'paired_comparison.png', dpi=300)
    plt.close()
    print("   Saved: paired_comparison.png")


def plot_feature_importance(
    hyper_imp_df: pd.DataFrame,
    pair_imp_df: pd.DataFrame,
    output_dir: Path,
    top_n: int = 15
):
    """Side-by-side horizontal bar charts of permutation importance."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, df, title, colour in zip(
        axes,
        [hyper_imp_df, pair_imp_df],
        [f'Top {top_n} Hypergraph Features', f'Top {top_n} Pairwise Features'],
        ['steelblue', 'gray']
    ):
        top = df.head(top_n)
        colors = [colour if v > 0 else 'lightcoral' for v in top['mean']]
        ax.barh(range(len(top)), top['mean'], xerr=top['std'],
                color=colors, edgecolor='black', capsize=3)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Mean Permutation Importance (PR-AUC drop)')
        ax.set_title(title)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.savefig(output_dir / 'feature_importance_comparison.png', dpi=300)
    plt.close()
    print("   Saved: feature_importance_comparison.png")


# =======================================================
# MAIN
# =======================================================

if __name__ == "__main__":

    start_time = time.time()
    print(f"Process started at {time.strftime('%H:%M:%S', time.localtime(start_time))}")

    # --- Output directory ---
    output_dir = CONFIG["BASE_OUTPUT_DIR"] / "essentiality_family_splits"
    output_dir.mkdir(parents=True, exist_ok=True)
    CONFIG["OUTPUT_DIR"] = output_dir

    print(f"\n{'='*70}")
    print(f"  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE")
    print(f"  Task   : Gene Essentiality")
    print(f"  Model  : {CONFIG['MODEL_TYPE']}")
    print(f"  Splits : pre-assigned family-level (protein_splits_all_strat.csv)")
    print(f"  Output : {output_dir}")
    print(f"{'='*70}\n")

    # --- Load data ---
    features_df = load_all_features()
    splits_df   = load_splits()

    split_indices = sorted(splits_df['split_index'].unique())
    print(f"\n   Running {len(split_indices)} splits: {split_indices}\n")

    # --- Resolve active features (only keep those actually present in features_df) ---
    hypergraph_features = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"]
                           if f in features_df.columns]
    pairwise_features   = [f for f in CONFIG["FEATURES"]["PAIRWISE"]
                           if f in features_df.columns]

    missing_hg   = [f for f in CONFIG["FEATURES"]["HYPERGRAPH"] if f not in features_df.columns]
    missing_pair = [f for f in CONFIG["FEATURES"]["PAIRWISE"]   if f not in features_df.columns]
    if missing_hg:
        print(f"   WARNING: {len(missing_hg)} hypergraph features not found in data: {missing_hg}")
    if missing_pair:
        print(f"   WARNING: {len(missing_pair)} pairwise features not found in data: {missing_pair}")

    print(f"   Active hypergraph features ({len(hypergraph_features)}):")
    for f in hypergraph_features:
        print(f"     - {f}")
    print(f"   Active pairwise features ({len(pairwise_features)}):")
    for f in pairwise_features:
        print(f"     - {f}")

    # --- Fill any NaNs in feature columns ---
    all_feature_cols = hypergraph_features + pairwise_features
    n_nans = features_df[all_feature_cols].isna().sum().sum()
    if n_nans > 0:
        print(f"   Filling {n_nans} missing feature values with 0.")
        features_df[all_feature_cols] = features_df[all_feature_cols].fillna(0)

    # --- Main loop over splits ---
    print(f"\n3. Running paired comparisons across {len(split_indices)} splits...\n")
    all_results = []

    for split_idx in split_indices:
        print(f"   Split {split_idx:>2}/{len(split_indices)}...", end=" ", flush=True)
        try:
            result = run_split(
                split_idx, features_df, splits_df,
                hypergraph_features, pairwise_features
            )
            all_results.append(result)
            winner = ("Hyper" if result['pr_auc_diff'] > 0
                      else "Pair" if result['pr_auc_diff'] < 0 else "Tie")
            print(f"train={result['n_train']} ({result['train_ess_pct']:.1f}% ess)  "
                  f"test={result['n_test']} ({result['test_ess_pct']:.1f}% ess)  |  "
                  f"Hyper: {result['hypergraph_pr_auc']:.4f}, "
                  f"Pair: {result['pairwise_pr_auc']:.4f}, "
                  f"Diff: {result['pr_auc_diff']:+.4f} [{winner}]")
        except Exception as e:
            print(f"ERROR: {e}")

    # --- Statistical comparison ---
    print("\n4. Statistical analysis...")
    stats = run_sign_test_comparison(all_results)
    print_statistical_summary(stats)

    # --- Plots ---
    print("\n5. Generating plots...")
    plot_paired_comparison(all_results, stats, output_dir)

    # --- Feature importance ---
    print("\n6. Aggregating feature importance...")
    hyper_imp_df = aggregate_feature_importance(all_results, 'hypergraph')
    pair_imp_df  = aggregate_feature_importance(all_results, 'pairwise')
    print_feature_importance_summary(hyper_imp_df, pair_imp_df, top_n=10)
    plot_feature_importance(hyper_imp_df, pair_imp_df, output_dir, top_n=15)

    # --- Save CSVs ---
    print("\n7. Saving outputs...")

    # Per-split summary (no nested dicts)
    summary_cols = ['split_index', 'n_train', 'n_test', 'train_ess_pct', 'test_ess_pct',
                    'hypergraph_pr_auc', 'hypergraph_f1',
                    'pairwise_pr_auc',   'pairwise_f1',
                    'pr_auc_diff',       'f1_diff']
    summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    summary_df.to_csv(output_dir / 'split_results.csv', index=False)
    print("   Saved: split_results.csv")

    # Per-protein predictions — hypergraph
    hyper_preds_all = pd.concat(
        [r['hypergraph_predictions'] for r in all_results], ignore_index=True
    )
    hyper_preds_all.to_csv(output_dir / 'hypergraph_predictions.csv', index=False)
    print("   Saved: hypergraph_predictions.csv")

    # Per-protein predictions — pairwise
    pair_preds_all = pd.concat(
        [r['pairwise_predictions'] for r in all_results], ignore_index=True
    )
    pair_preds_all.to_csv(output_dir / 'pairwise_predictions.csv', index=False)
    print("   Saved: pairwise_predictions.csv")

    # Feature importance
    hyper_imp_df.to_csv(output_dir / 'hypergraph_feature_importance.csv', index=False)
    pair_imp_df.to_csv(output_dir / 'pairwise_feature_importance.csv', index=False)
    print("   Saved: hypergraph_feature_importance.csv")
    print("   Saved: pairwise_feature_importance.csv")

    with open(output_dir / 'statistical_summary.txt', 'w') as f:
            f.write("PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE\n")
            f.write("Task: Gene Essentiality\n")
            f.write(f"Model: {CONFIG['MODEL_TYPE']}\n")
            f.write(f"Number of splits: {stats['n_runs']}\n\n")
            f.write(f"Hypergraph features ({len(hypergraph_features)}):\n")
            for feat in hypergraph_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nPairwise features ({len(pairwise_features)}):\n")
            for feat in pairwise_features:
                f.write(f"  - {feat}\n")
            f.write(f"\nHypergraph PR-AUC: {stats['hypergraph_pr_auc_mean']:.4f} ± {stats['hypergraph_pr_auc_std']:.4f}\n")
            f.write(f"Pairwise PR-AUC:   {stats['pairwise_pr_auc_mean']:.4f} ± {stats['pairwise_pr_auc_std']:.4f}\n\n")
            f.write(f"Mean Difference:   {stats['mean_difference']:.4f} ± {stats['std_difference']:.4f}\n")
            f.write(f"Hypergraph wins:   {stats['hypergraph_wins']}/{stats['n_runs']}\n")
            f.write(f"Pairwise wins:     {stats['pairwise_wins']}/{stats['n_runs']}\n")
            f.write(f"Ties:              {stats['ties']}/{stats['n_runs']}\n\n")
            f.write(f"Sign test p (one-sided, H > P): {stats['sign_test_p_greater']:.6f}\n")
            f.write(f"Sign test p (two-sided):        {stats['sign_test_p_two_sided']:.6f}\n")

    print(f"\n{'='*70}")
    print("  COMPLETE")
    print(f"{'='*70}")

    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

Process started at 18:00:05

  PAIRED COMPARISON: HYPERGRAPH vs PAIRWISE
  Task   : Gene Essentiality
  Model  : XGBoost
  Splits : pre-assigned family-level (protein_splits_all_strat.csv)
  Output : xgboost/two_hop_no_unique/essentiality_family_splits

1. Loading feature data...
   Hypergraph features shape : (3394, 35)
   Pairwise features shape   : (3394, 9)
   Combined shape            : (3394, 43)
2. Loading pre-assigned splits...
   Splits file rows  : 50910
   Unique proteins   : 3394
   Number of splits  : 15
   Labelled proteins : 3280  (23.6% essential)

   Running 15 splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

   Active hypergraph features (12):
     - base_Degree
     - base_TriangleCount
     - base_UniquePartners
     - base_AvgNeighbourDegree
     - stoich_WeightedTriangles
     - stoich_AvgNeighbourDegreeS

<ipython-input-40-bc15c3af89bd>:504: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([pair_vals, hyper_vals],


   Saved: paired_comparison.png

6. Aggregating feature importance...

  FEATURE IMPORTANCE (Permutation — PR-AUC drop)

  Top 10 Hypergraph Features:
  Rank   Feature                             Mean         Std       
  -----------------------------------------------------------------
  1      base_TriangleCount                  0.2009       0.0519
  2      base_AvgNeighbourDegree             0.1036       0.0272
  3      stoich_WeightedTriangles            0.0682       0.0686
  4      base_UniquePartners                 0.0302       0.0361
  5      stoich_AvgNeighbourDegreeStoich     0.0177       0.0340
  6      base_Degree                         0.0160       0.0221
  7      stoich_RangeComplexSize             0.0103       0.0169
  8      stoich_MedianRatio                  0.0080       0.0127
  9      protein_NumSubgroups                0.0047       0.0125
  10     protein_AvgSubgroupSize             0.0014       0.0024

  Top 10 Pairwise Features:
  Rank   Feature                 